# Redox FHIR Pipeline - Silver Tables

This notebook creates Silver-layer streaming tables for each FHIR resource type, pivoting the exploded key-value pairs into structured columns.

## Dynamic Table Creation
For each resource type discovered in `resource_schemas`, this notebook:
1. Retrieves the list of columns for that resource type
2. Creates a streaming table with those columns as VARIANT fields
3. Establishes foreign key relationship to `bundle_meta`

## Usage
Run this notebook with the `resource_type` parameter to create a specific resource table:
```
resource_type = 'Patient'
```

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE resource_type STRING DEFAULT 'Patient';
DECLARE OR REPLACE VARIABLE full_refresh BOOLEAN DEFAULT FALSE;

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE schema_use = COALESCE(:schema_use, schema_use);
SET VARIABLE resource_type = COALESCE(:resource_type, resource_type);
SET VARIABLE full_refresh = COALESCE(:full_refresh = 'true', full_refresh);

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT 
  current_catalog() AS catalog, 
  current_schema() AS schema,
  resource_type AS target_resource,
  full_refresh;

## Step 1: Check for Schema Changes

If new columns are detected for this resource type, we need to perform a full refresh to add them.

In [ ]:
-- Check if table exists and if there are new columns
DECLARE OR REPLACE VARIABLE new_columns_detected BOOLEAN DEFAULT FALSE;

SET VARIABLE new_columns_detected = (
  WITH new_columns AS (
    SELECT column_name 
    FROM resource_schemas 
    WHERE resource_type = :resource_type
    EXCEPT
    SELECT column_name 
    FROM IDENTIFIER(catalog_use || '.information_schema.columns')
    WHERE 
      table_catalog = catalog_use
      AND table_schema = schema_use
      AND table_name = LOWER(resource_type)
  )
  SELECT COALESCE(COUNT(*) > 0, TRUE) FROM new_columns
);

SELECT new_columns_detected AS schema_evolution_detected;

In [ ]:
-- Drop table if full refresh requested or schema evolved
DECLARE OR REPLACE VARIABLE drop_stmt STRING;

SET VARIABLE drop_stmt = CASE 
  WHEN (full_refresh OR new_columns_detected) 
  THEN 'DROP TABLE IF EXISTS ' || resource_type || ';'
  ELSE 'SELECT ''Incremental refresh for ' || resource_type || ''' AS status;'
END;

SELECT drop_stmt;
EXECUTE IMMEDIATE drop_stmt;

## Step 2: Get Column List for Resource Type

In [ ]:
-- Get ordered list of columns for PIVOT
DECLARE OR REPLACE VARIABLE resource_columns STRING;

SET VARIABLE resource_columns = (
  WITH existing_columns AS (
    SELECT 
      column_name,
      ordinal_position
    FROM IDENTIFIER(catalog_use || '.information_schema.columns')
    WHERE 
      table_catalog = catalog_use
      AND table_schema = schema_use
      AND table_name = LOWER(resource_type)
  ),
  ordered_columns AS (
    SELECT
      rs.column_name,
      ec.ordinal_position
    FROM resource_schemas rs
    LEFT JOIN existing_columns ec ON rs.column_name = ec.column_name
    WHERE rs.resource_type = :resource_type
    ORDER BY ec.ordinal_position NULLS LAST, rs.column_name
  )
  SELECT ARRAY_JOIN(COLLECT_LIST(column_name), "', '") FROM ordered_columns
);

SELECT resource_columns AS columns_for_pivot;

## Step 3: Create Silver Streaming Table

In [ ]:
-- Build and execute the CREATE STREAMING TABLE statement
DECLARE OR REPLACE VARIABLE create_silver_stmt STRING;

SET VARIABLE create_silver_stmt = "
CREATE OR REFRESH STREAMING TABLE " || resource_type || " (
  " || LOWER(resource_type) || "_uuid STRING NOT NULL COMMENT 'Unique resource identifier'
  ,bundle_uuid STRING NOT NULL COMMENT 'Reference to parent bundle'
  ," || LOWER(resource_type) || "_url STRING COMMENT 'FHIR fullUrl for this resource'
  ,CONSTRAINT fk_" || LOWER(resource_type) || "_bundle 
    FOREIGN KEY (bundle_uuid) REFERENCES bundle_meta(bundle_uuid)
)
COMMENT 'Silver layer - Parsed " || resource_type || " resources from Redox FHIR bundles'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'silver'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS SELECT
  uuid() AS " || LOWER(resource_type) || "_uuid
  ,*
FROM (
  SELECT
    bundle_uuid
    ,full_url AS " || LOWER(resource_type) || "_url
    ,key
    ,value
  FROM STREAM(resources_exploded)
  WHERE resource_type = '" || resource_type || "'
)
PIVOT (
  FIRST(value) FOR key IN ('" || resource_columns || "')
);
";

SELECT create_silver_stmt AS create_statement;

In [ ]:
EXECUTE IMMEDIATE create_silver_stmt;

In [ ]:
-- Verify the created table
DECLARE OR REPLACE VARIABLE verify_stmt STRING;

SET VARIABLE verify_stmt = 'SELECT * FROM ' || resource_type || ' LIMIT 5;';

EXECUTE IMMEDIATE verify_stmt;

In [ ]:
-- Show table schema
DECLARE OR REPLACE VARIABLE describe_stmt STRING;

SET VARIABLE describe_stmt = 'DESCRIBE TABLE ' || resource_type || ';';

EXECUTE IMMEDIATE describe_stmt;